# Fase 4 - Entrenamiento baseline DenseNet121

Notebook para entrenar el baseline DenseNet121 en Kaggle usando los CSV validados en Fase 2 y Fase 3. No implementa CNN-ViT, Grad-CAM ni inferencia de API.

## 1. Verificar GPU

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. git pull

In [ ]:
# Si el repo no esta en el entorno, descomenta y ajusta la URL:
# !git clone https://github.com/USUARIO/Tesis_CXR.git /kaggle/working/Tesis_CXR

candidate_repos = [Path.cwd(), Path('/kaggle/working/Tesis_CXR')]
REPO_DIR = next((path for path in candidate_repos if (path / 'scripts').exists()), None)

if REPO_DIR is None:
    raise FileNotFoundError('No se encontro el repositorio. Clona Tesis_CXR en /kaggle/working/Tesis_CXR.')

print('Repo:', REPO_DIR)
subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

## 3. Confirmar existencia de train.csv y val.csv

In [ ]:
TRAIN_CSV = Path('/kaggle/working/processed/splits/train.csv')
VAL_CSV = Path('/kaggle/working/processed/splits/val.csv')
LABELS_JSON = REPO_DIR / 'artifacts' / 'labels.json'
OUTPUT_DIR = Path('/kaggle/working/artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [TRAIN_CSV, VAL_CSV, LABELS_JSON]:
    if not path.exists():
        raise FileNotFoundError(path)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
print('Train rows:', len(train_df))
print('Val rows:', len(val_df))
assert 'No Finding' not in train_df.columns
assert 'No Finding' not in val_df.columns

## 4. Ejecutar check_dataset.py

In [ ]:
check_cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'check_dataset.py'),
    '--csv-path', str(TRAIN_CSV),
    '--labels-json', str(LABELS_JSON),
    '--batch-size', '8',
    '--num-workers', '2',
]
subprocess.run(check_cmd, check=True)

## 5. Ejecutar train_baseline.py

In [ ]:
train_cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'train_baseline.py'),
    '--train-csv', str(TRAIN_CSV),
    '--val-csv', str(VAL_CSV),
    '--labels-json', str(LABELS_JSON),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '3',
    '--batch-size', '32',
    '--num-workers', '2',
    '--lr', '1e-4',
    '--image-size', '224',
    '--device', 'auto',
]
subprocess.run(train_cmd, check=True)

## 6. Mostrar baseline_metrics.json

In [ ]:
metrics_path = OUTPUT_DIR / 'baseline_metrics.json'
with metrics_path.open('r', encoding='utf-8') as file:
    metrics = json.load(file)

metrics

## 7. Mostrar baseline_training_history.csv

In [ ]:
history_path = OUTPUT_DIR / 'baseline_training_history.csv'
history_df = pd.read_csv(history_path)
history_df

## 8. Confirmar baseline_densenet121_best.pt

In [ ]:
best_model_path = OUTPUT_DIR / 'baseline_densenet121_best.pt'
assert best_model_path.exists()
print('Best model:', best_model_path)
print('Size MB:', best_model_path.stat().st_size / (1024 * 1024))